In [ ]:
# =========================================================
# LVBERT OOF SEMANTIC SIGNAL PIPELINE FOR HSSC
# Compatible with newer transformers versions:

# Run this in Colab / Jupyter to generate:
#   - trainval_oof_predictions.csv
#   - lvbert_test_predictions.csv
# Then run your HSSC feature merging pipeline
# =========================================================

# !pip install -q transformers datasets accelerate sentencepiece

import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)
from datasets import Dataset

# =========================================================
# 0. SETTINGS
# =========================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
set_seed(RANDOM_SEED)

TRAIN_CSV = "/content/train.csv"
VAL_CSV   = "/content/validation.csv"
TEST_CSV  = "/content/test.csv"

TITLE_COL = "title"
TEXT_COL  = "full_text"
LABEL_COL = "class_label"

MODEL_NAME = "AiLab-IMCS-UL/lvbert"

OUTPUT_DIR = "/content/lvbert_oof_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Best hyperparameters from previous experiment
MAX_LEN = 256
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 1
N_FOLDS = 5

# =========================================================
# 1. LOAD DATA
# =========================================================
train_df = pd.read_csv(TRAIN_CSV, encoding="utf-8")
val_df   = pd.read_csv(VAL_CSV, encoding="utf-8")
test_df  = pd.read_csv(TEST_CSV, encoding="utf-8")

for df in [train_df, val_df, test_df]:
    df[TITLE_COL] = df[TITLE_COL].fillna("").astype(str)
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
    df["input_text"] = df[TITLE_COL] + " [SEP] " + df[TEXT_COL]

for df in [train_df, val_df, test_df]:
    if LABEL_COL in df.columns:
        df[LABEL_COL] = df[LABEL_COL].astype(int)

trainval_df = pd.concat([train_df, val_df], ignore_index=True)

n_train = len(train_df)
n_val = len(val_df)
n_test = len(test_df)

print(f"Train: {n_train}, Val: {n_val}, TrainVal: {len(trainval_df)}, Test: {n_test}")

# =========================================================
# 2. TOKENIZER AND HELPERS
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["input_text"],
        truncation=True,
        padding=False,
        max_length=MAX_LEN
    )

def make_hf_dataset(df, with_labels=True):
    data = {"input_text": df["input_text"].tolist()}
    if with_labels and LABEL_COL in df.columns:
        data["label"] = df[LABEL_COL].tolist()

    ds = Dataset.from_dict(data)
    ds = ds.map(tokenize_function, batched=True, remove_columns=["input_text"])
    return ds

def get_positive_class_probs(trainer, dataset):
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    return probs[:, 1]

def make_training_args(output_subdir, epochs=NUM_EPOCHS):
    return TrainingArguments(
        output_dir=output_subdir,
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=RANDOM_SEED,
        logging_steps=50,
        report_to="none",
        save_total_limit=1
    )

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================================================
# 3. OOF PREDICTIONS ON TRAINVAL (5-FOLD CV)
# =========================================================
print("\n=== STARTING OOF PREDICTIONS ===")

labels_trainval = trainval_df[LABEL_COL].values
oof_probs = np.zeros(len(trainval_df), dtype=np.float32)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

for fold, (train_idx, valid_idx) in enumerate(skf.split(trainval_df, labels_trainval), start=1):
    print(f"\n--- Fold {fold}/{N_FOLDS} ---")

    fold_train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
    fold_valid_df = trainval_df.iloc[valid_idx].reset_index(drop=True)

    fold_train_ds = make_hf_dataset(fold_train_df, with_labels=True)
    fold_valid_ds = make_hf_dataset(fold_valid_df, with_labels=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    fold_output_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    training_args = make_training_args(fold_output_dir)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=fold_train_ds,
        eval_dataset=fold_valid_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )
        ]
    )

    trainer.train()

    fold_probs = get_positive_class_probs(trainer, fold_valid_ds)
    oof_probs[valid_idx] = fold_probs

    fold_pred_labels = (fold_probs >= 0.5).astype(int)
    fold_acc = accuracy_score(labels_trainval[valid_idx], fold_pred_labels)
    fold_auc = roc_auc_score(labels_trainval[valid_idx], fold_probs)

    print(f"Fold {fold} - Accuracy: {fold_acc:.4f}, ROC-AUC: {fold_auc:.4f}")

    del model
    del trainer
    cleanup()

# =========================================================
# 4. SAVE OOF PREDICTIONS
# =========================================================
trainval_oof_df = trainval_df[["input_text", LABEL_COL]].copy()
trainval_oof_df["pred_prob"] = oof_probs
trainval_oof_df["pred_label"] = (oof_probs >= 0.5).astype(int)

oof_acc = accuracy_score(labels_trainval, trainval_oof_df["pred_label"])
oof_auc = roc_auc_score(labels_trainval, oof_probs)

print(f"\nOOF overall - Accuracy: {oof_acc:.4f}, ROC-AUC: {oof_auc:.4f}")

oof_path = os.path.join(OUTPUT_DIR, "trainval_oof_predictions.csv")
trainval_oof_df.to_csv(oof_path, index=False, encoding="utf-8")
print(f"Saved: {oof_path}")

# =========================================================
# 5. TRAIN FINAL MODEL ON FULL TRAINVAL
# =========================================================
print("\n=== TRAINING FINAL MODEL ON FULL TRAINVAL ===")

full_train_ds = make_hf_dataset(trainval_df, with_labels=True)
val_eval_ds = make_hf_dataset(val_df, with_labels=True)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

final_output_dir = os.path.join(OUTPUT_DIR, "final_model")
final_training_args = make_training_args(final_output_dir)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_train_ds,
    eval_dataset=val_eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ]
)

final_trainer.train()

# =========================================================
# 6. TEST PREDICTIONS
# =========================================================
print("\n=== GENERATING TEST PREDICTIONS ===")

test_has_labels = LABEL_COL in test_df.columns
test_ds = make_hf_dataset(test_df, with_labels=test_has_labels)

test_probs = get_positive_class_probs(final_trainer, test_ds)
test_pred_labels = (test_probs >= 0.5).astype(int)

if test_has_labels:
    test_pred_df = test_df[["input_text", LABEL_COL]].copy()
else:
    test_pred_df = test_df[["input_text"]].copy()

test_pred_df["pred_prob"] = test_probs
test_pred_df["pred_label"] = test_pred_labels

if test_has_labels:
    def assign_group(row):
        if row[LABEL_COL] == 1 and row["pred_label"] == 1:
            return "TP"
        elif row[LABEL_COL] == 0 and row["pred_label"] == 0:
            return "TN"
        elif row[LABEL_COL] == 0 and row["pred_label"] == 1:
            return "FP"
        else:
            return "FN"

    test_pred_df["group"] = test_pred_df.apply(assign_group, axis=1)

    test_labels = test_df[LABEL_COL].values
    test_acc = accuracy_score(test_labels, test_pred_labels)
    test_auc = roc_auc_score(test_labels, test_probs)

    print(f"\nTest - Accuracy: {test_acc:.4f}, ROC-AUC: {test_auc:.4f}")

test_path = os.path.join(OUTPUT_DIR, "lvbert_test_predictions.csv")
test_pred_df.to_csv(test_path, index=False, encoding="utf-8")
print(f"Saved: {test_path}")

# =========================================================
# 7. SUMMARY
# =========================================================
print("\n=== FILES READY FOR HSSC PIPELINE ===")
print(f"OOF predictions:  {oof_path}")
print(f"Test predictions: {test_path}")

print("\nNext step:")
print("1. Run your feature merging code (HSSC merged features)")
print("2. Update the file paths to:")
print(f"   TRAINVAL_OOF_FILE = '{oof_path}'")
print(f"   TEST_PRED_FILE    = '{test_path}'")

Train: 1429, Val: 306, TrainVal: 1735, Test: 307

=== STARTING OOF PREDICTIONS ===

--- Fold 1/5 ---


Map:   0%|          | 0/1388 [00:00<?, ? examples/s]

Map:   0%|          | 0/347 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: AiLab-IMCS-UL/lvbert
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


=== GENERATING TEST PREDICTIONS ===


Map:   0%|          | 0/307 [00:00<?, ? examples/s]


Test - Accuracy: 0.9577, ROC-AUC: 0.9917
Saved: /content/lvbert_oof_output/lvbert_test_predictions.csv

=== FILES READY FOR HSSC PIPELINE ===
OOF predictions:  /content/lvbert_oof_output/trainval_oof_predictions.csv
Test predictions: /content/lvbert_oof_output/lvbert_test_predictions.csv

Next step:
1. Run your feature merging code (HSSC merged features)
2. Update the file paths to:
   TRAINVAL_OOF_FILE = '/content/lvbert_oof_output/trainval_oof_predictions.csv'
   TEST_PRED_FILE    = '/content/lvbert_oof_output/lvbert_test_predictions.csv'
